# Sprint stage 1: reconstruct rooms from phone videos

Turns each room video into wall planes plus a labelled top-down plot (`walls.png`), using **MapAnything with the Apache-licensed weights**. Results go to Google Drive so they survive the session.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free) → Save.

**What you need in Drive:** a folder `MyDrive/to3d-sprint/videos/` with one video per room, named by room id, e.g. `r1.mp4`, `r2.mov`. The file name (without extension) becomes the `room_id` in `tape.csv`.

**Privacy (rule G5):** extracted frames are deleted after each room; only geometry is kept. The last cell can delete the videos too.

Run the cells top to bottom. Steps 1–3 take ~5 min once; each room then takes a few minutes.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU, then rerun."
print("torch", torch.__version__, "| GPU OK")

## 2. Mount Google Drive and set folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
SPRINT = Path('/content/drive/MyDrive/to3d-sprint')   # change if you use another folder
VIDEOS = SPRINT / 'videos'
ROOMS  = SPRINT / 'rooms'
for p in (VIDEOS, ROOMS):
    p.mkdir(parents=True, exist_ok=True)
vids = sorted(v for v in VIDEOS.iterdir() if v.suffix.lower() in ('.mp4', '.mov', '.m4v', '.3gp', '.mkv'))
print(f"{len(vids)} video(s) in {VIDEOS}:", [v.name for v in vids])
if not vids:
    print("Put your room videos in that Drive folder (e.g. r1.mp4), then rerun this cell.")

## 3. Install MapAnything and get `stage1.py`

In [ ]:
%cd /content
!test -d map-anything || git clone -q https://github.com/facebookresearch/map-anything.git
!pip install -q -e map-anything
!pip install -q opencv-python-headless matplotlib

`stage1.py` is fetched from your GitHub repo. If the repo is private the download fails, and the cell asks you to upload `creating-idea/09-sprint/stage1.py` from your computer instead (or put it in `MyDrive/to3d-sprint/`).

In [ ]:
import shutil, subprocess
%cd /content
URL = 'https://raw.githubusercontent.com/Revaant-D/to3D/claude/determined-heisenberg-3tsudd/creating-idea/09-sprint/stage1.py'
ok = subprocess.run(['wget', '-q', '-O', 'stage1.py', URL]).returncode == 0 and os.path.getsize('stage1.py') > 1000
if not ok and (SPRINT / 'stage1.py').exists():
    shutil.copy(SPRINT / 'stage1.py', 'stage1.py'); ok = True
if not ok:
    from google.colab import files
    print("Download failed (private repo?). Upload stage1.py now:")
    up = files.upload()
    name = next(iter(up)); os.rename(name, 'stage1.py') if name != 'stage1.py' else None
!python stage1.py --help | head -5

## 4. Reconstruct every room

Skips rooms that already have `walls.json` in Drive (set `REDO = True` to redo them). If you hit a CUDA out-of-memory error, lower `FRAMES` to 40 or 30.

In [ ]:
FRAMES = 60      # keyframes per video; lower if GPU memory runs out
REDO = False

import subprocess, sys
done, failed = [], []
for v in vids:
    room = v.stem
    if (ROOMS / room / 'walls.json').exists() and not REDO:
        print(f"skip {room} (already done)"); continue
    print(f"\n=== {room} ({v.name}) ===")
    r = subprocess.run([sys.executable, 'stage1.py', 'reconstruct', '--room', room, '--video', str(v),
                        '--rooms-dir', str(ROOMS), '--frames', str(FRAMES)],
                       capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-3000:]); failed.append(room)
    else:
        done.append(room)
print("\ndone:", done, "| failed:", failed)

## 5. Look at the plots and note the wall IDs

For each room, find the two walls you taped between and write their IDs (`W0`, `W1`, …) into `wall_a` / `wall_b` in your `tape.csv`. Furniture fronts also show up as planes; use them only if you taped to them.

Check the line under each plot: `up consistency` should be ≥ 0.8, and a real wall usually has rms ≤ 1–2 cm.

In [ ]:
import json
from IPython.display import Image, display
for d in sorted(p for p in ROOMS.iterdir() if (p / 'walls.png').exists()):
    meta = json.loads((d / 'walls.json').read_text())
    print(f"--- {d.name}: up consistency {meta['up_consistency']:.2f}, {len(meta['walls'])} planes")
    for w in meta['walls']:
        print(f"   {w['id']}: extent {w['extent_m']:.2f} m, rms {w['rms_m']*100:.1f} cm, {w['inliers']} pts")
    display(Image(str(d / 'walls.png'), width=600))

## 6. (Optional) Evaluate here too

If you've filled in `tape.csv` and uploaded it to `MyDrive/to3d-sprint/tape.csv`, this runs the evaluation and shows the KILL/PASS verdict. You can also run `evaluate` on your laptop after downloading the `rooms/` folder.

In [ ]:
TAPE = SPRINT / 'tape.csv'
if TAPE.exists():
    !python stage1.py evaluate --tape "{TAPE}" --rooms-dir "{ROOMS}" --out "{SPRINT / 'out'}" --tolerance-cm 2
else:
    print(f"No {TAPE} yet: fill in tape_template.csv, save as tape.csv, upload it there.")

## 7. (Optional) Delete the videos once all rooms are done (G5)

In [ ]:
DELETE_VIDEOS = False   # set True and rerun to delete the videos from Drive
if DELETE_VIDEOS:
    for v in vids:
        v.unlink(); print("deleted", v.name)
else:
    print("Videos kept. Set DELETE_VIDEOS = True when you no longer need them.")